In [6]:
#
# Import all dependencies
#
import os
import time
import logging
import warnings
import traceback
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from joblib import Parallel, delayed
from nested_pandas import NestedDtype
from joblib import wrap_non_picklable_objects
import os
import numpy as np
import tensorflow as tf 
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from astra.bands.bands import ztf_band
from astra.utils.helper import filter_by_class

In [7]:
def create_dataset(df,
                    target,
                    njobs=1,
                    bands=None,
                    seed=42,
                    label=None,
                    lookup_class=None,
                    min_detec=100,
                    train_size=0.80,
                    max_lcs_per_chunk=100,
                    partition_info=None):
    
    """
    Create tf.records from the catalog

    Parameters: 
    ---------------------------------------------------------
        df (DataFrame): contains the catalog file
        target (str): directory path for the files to be stored
        bands (dict): band information (filters should be in a ordered sequence)
                        e.g. ZTF band order is {g, r, i}, meaning g-filter 
                        has the highest priority and i-filter has the lowest.
        n_jobs (int): # of parallel jobs
        label (str): label associated with the catalog.
                        Provide this value only if the 'Class' column 
                        is missing in the dataframe (df). 
        train_size (float): train fraction
        min_detec (int): minimum detections in each light curve
                        Note: For ZTF, "min_detec" for g & r-filters is user-defined
                        but, for i-filter there is no min_detec 
                        (it's set to 0 based on the filter order in the bands dict)
        max_lcs_per_chunk (int): # of lcs to be stored in a chunk
        
    """
    #
    # First partition_info in None
    # Start from the 2nd partition_info
    #
    if partition_info is not None:
        info_df = pd.DataFrame()
        LC_COLUMN = "lc"
        n_partition = partition_info['number']
        str_div = partition_info['division']
        #
        #
        #
        try: 
            if not train_size or train_size > 1:
                raise ValueError(f"Please provide a valid train_size fraction (between 0-1). Got {train_size}")
            
            if bands is None:
                raise ValueError(f"Please provide band information with ordered filters. Got {bands}")
            
            if not os.path.exists(target):
                os.makedirs(target, exist_ok=True)
            #
            dest = os.path.join(target, "objects_test")
            os.makedirs(dest, exist_ok=True)
            #
            df = df.assign(**{LC_COLUMN: df[LC_COLUMN].astype(NestedDtype.from_pandas_arrow_dtype(df.dtypes[LC_COLUMN]))},)
            df = df.dropna(subset=['lc'])
            #
            #
            #
            if "Class" not in df.columns:
                if label: 
                    #
                    #
                    #
                    df['Class'] = label
                else:
                    raise AttributeError(f"\nException Raised: You must provide a class/label to this catalog."
                        f"\nThe 'Class' column couldn't be inferred from the catalog and the 'label'" 
                        f"\nparameter is {label} . Please provide a 'Class' column to the catalog or define" 
                        f"\nthe 'label' parameter.")
            
            #
            # Filter by lookup_class if provided -- otherwise use all classes
            #
            if lookup_class is not None:
                df = df[df['Class'].isin(lookup_class.keys())]
            #
            
            # cls_groups = df.groupby('Class')
            #
            # Save the number of classes and their counts in a .CSV file
            #
            unique, counts = np.unique(df['Class'], return_counts=True)
            info_df['label'] = unique
            info_df['size'] = counts
            info_df['start_index'] = str_div
            info_df.to_csv(os.path.join(target, "objects_test", f'partition_{n_partition}.csv'), index=False)
            # -----------------------------------
            # Separate by class
            # -----------------------------------
            
            #
            # Write lcs in the records
            #
            # for cls_name, cls_meta in tqdm(cls_groups, total=len(cls_groups)):
            #     subsets = train_test_split(cls_meta, train_size=train_size, seed=seed)

            #     for subset_name, frame in subsets:
            #         if frame is None:
            #             continue
            #         dest = os.path.join(target, subset_name, f"partition_{n_partition}", cls_name)
            #         os.makedirs(dest, exist_ok=True)
            #         write_records(frame, dest, max_lcs_per_chunk, n_jobs=njobs, 
            #                       bands=bands, min_detec=min_detec, n_partition=n_partition)

        except Exception:
            print(f"\n\n[Traceback]\n {traceback.format_exc()}\n")
            

In [9]:
#
# Read catalog
#
read_catalog = read_hats('/media3/majumder/dataset/lyrae/hats/zubercal_vrrlyr', )
path_to_store="/media3/majumder/dataset/lyrae/"
#
#
#
catalog_compute = read_catalog._ddf.map_partitions(create_dataset, 
                                                        target=path_to_store,
                                                        bands=ztf_band,
                                                        lookup_class={'RRab':200, 'RRd':100},
                                                        seed=42,
                                                        min_detec=100,
                                                        train_size=.80,
                                                        max_lcs_per_chunk=100)

with Client() as client:
    catalog_compute.compute(scheduler='processes')


print("\nDone!")


Done!


In [16]:
target_counts = pd.read_csv("./overall_label_counts.csv").set_index('Label')['Count'].to_dict()
target_counts = {key.encode('utf-8'): value for key, value in target_counts.items()}
source = "/media3/majumder/dataset/cepheids_new/test/"
target_counts

{b'ACEP': 62, b'DCEP': 2379, b'T2CEP': 617}

In [17]:
# --- Main execution ---
# In your code, you would use your real 'filenames' dataset:
# final_filtered_dataset = filter_by_class_counts(filenames, target_counts, _parse_function)
final_filtered_dataset = filter_by_class(source, target_counts)


print(final_filtered_dataset)


# # --- Verification (Optional but Recommended) ---
# # Let's check if the final dataset has the correct counts.
# class_counts = {k: 0 for k in target_counts.keys()}
# for sample in final_filtered_dataset:
#     label = sample['label'].numpy()
#     if label in class_counts:
#         class_counts[label] += 1

# print("\n--- Verification of Final Counts ---")
# for class_name, count in class_counts.items():
#     print(f"Class {class_name.decode()}: Found {count} samples (Target: {target_counts[class_name]})")

Filtering for each class...
 - Getting 62 samples for class: ACEP
 - Getting 2379 samples for class: DCEP
 - Getting 617 samples for class: T2CEP

Successfully created a dataset with 3058 total samples.
<_PrefetchDataset element_spec={'id': TensorSpec(shape=(), dtype=tf.int64, name=None), 'last_index': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'label': TensorSpec(shape=(), dtype=tf.string, name=None), 'bands': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'input_id': TensorSpec(shape=(None, 4), dtype=tf.float64, name=None)}>


In [8]:
# 1. Set the path to the folder containing the CSV files
#    Replace 'path/to/your/csv_folder' with the actual path to your folder.
folder_path = "/media3/majumder/dataset/cepheids/objects/"

# 2. Get a list of all CSV files in the folder
all_files = glob.glob(folder_path + "/*.csv")

# 3. Create an empty list to hold the individual DataFrames
list_of_dfs = []

# 4. Loop through the list of files, read each one into a DataFrame,
#    and append it to the list.
#    Assuming the column with labels is named 'label' in all CSVs.
#    If your label column has a different name, change 'label' accordingly.
for filename in all_files:
    df = pd.read_csv(filename)
    # Check if 'label' and 'count' columns exist
    if 'label' in df.columns and 'size' in df.columns:
        list_of_dfs.append(df)
    else:
        print(f"Skipping {filename} as it does not contain 'label' and 'size' columns.")


# 5. Concatenate all the DataFrames in the list into a single DataFrame
if list_of_dfs:
    combined_df = pd.concat(list_of_dfs, ignore_index=True)

    # 6. Group by the 'label' column and sum the 'size' for each label
    label_counts = combined_df.groupby('label')['size'].sum().reset_index()

    # 7. Rename the columns for clarity
    label_counts.columns = ['Label', 'Count']

    # 8. Set the path and filename for the output CSV file
    output_filename = 'overall_label_counts.csv'

    # 9. Save the aggregated counts to a new CSV file
    label_counts.to_csv(output_filename, index=False)

    print(f"Successfully created {output_filename} with the overall label counts.")
else:
    print("No valid CSV files were found to process.")

Successfully created overall_label_counts.csv with the overall label counts.
